# 4. Camada Silver

A camada Silver representa a etapa de padronização e integração dos dados históricos de produção offshore armazenados na camada Bronze.

A avaliação de qualidade realizada anteriormente identificou diferenças de nomenclatura e tipos de dados entre os períodos, além de características que precisam ser consideradas durante a consolidação do histórico.

Com base nas regras de tratamento definidas no notebook `03_qualidade_dados`, esta etapa realizará as transformações necessárias para produzir uma representação única e consistente dos dados entre 2005 e 2025.

As principais transformações previstas são:

- padronização dos nomes das colunas;
- definição explícita dos tipos de dados;
- conversão dos atributos quantitativos;
- tratamento do atributo temporal;
- consolidação das 11 tabelas históricas;
- persistência dos dados tratados em formato Delta;
- validação da camada Silver.

Os tratamentos serão realizados preservando os valores nulos e os registros identificados durante a avaliação de qualidade, evitando alterações sem justificativa nos dados fornecidos pela fonte.

## 4.1 Objetivo da camada Silver

O objetivo desta etapa é transformar as 11 tabelas da camada Bronze em uma estrutura histórica única, padronizada e adequada para utilização nas etapas posteriores de modelagem e análise.

Ao final do processo, os dados referentes ao período de **2005 a 2025** serão consolidados em uma tabela Delta na camada Silver, mantendo a granularidade observada de **período, poço, campo e instalação**.

A tabela resultante servirá como principal fonte para a construção do modelo analítico da camada Gold.

## 4.2 Preparação da camada Silver

Antes da aplicação das transformações, será criado o schema `silver` no catálogo `workspace`.

Esse schema será utilizado para armazenar os dados tratados e consolidados, mantendo a separação lógica entre os dados provenientes da ingestão, armazenados na Bronze, e os dados padronizados utilizados nas etapas analíticas posteriores.

In [0]:
# Cria o schema destinado às tabelas tratadas e consolidadas
# da camada Silver.

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.silver
""")

display(
    spark.sql("SHOW SCHEMAS IN workspace")
)

databaseName
bronze
default
gold
information_schema
silver


## 4.3 Padronização dos nomes das colunas

A análise estrutural demonstrou que os arquivos referentes aos períodos de 2022 a 2025 apresentam seus cabeçalhos delimitados por colchetes, enquanto os períodos anteriores utilizam os nomes originais sem essa delimitação.

Além de eliminar essa diferença, os nomes técnicos das colunas serão padronizados utilizando a convenção `snake_case`, sem espaços ou acentos.

As unidades de medida serão preservadas nos nomes dos atributos quantitativos de forma simplificada, permitindo distinguir corretamente grandezas expressas em `m³` e `Mm³`.

Essa padronização será aplicada apenas à camada Silver. As tabelas Bronze permanecerão inalteradas.

### 4.3.1 Definição do mapeamento das colunas

Para garantir que todos os períodos utilizem exatamente a mesma nomenclatura, será definido explicitamente o mapeamento entre os 21 atributos presentes na fonte e seus respectivos nomes técnicos na camada Silver.

A definição explícita do mapeamento também evita que diferenças de caracteres especiais ou unidades de medida produzam nomes distintos durante a consolidação das tabelas.

In [0]:
# Define a nomenclatura padronizada que será utilizada
# em todo o histórico consolidado da camada Silver.

mapa_colunas = {
    "Ano": "ano",
    "Mês/Ano": "mes_ano",
    "Estado": "estado",
    "Bacia": "bacia",
    "Campo": "campo",
    "Poço": "poco",
    "Ambiente": "ambiente",
    "Instalação": "instalacao",
    "Produção de Óleo (m³)": "producao_oleo_m3",
    "Produção de Condensado (m³)": "producao_condensado_m3",
    "Produção de Gás Associado (Mm³)": "producao_gas_associado_mm3",
    "Produção de Gás Não Associado (Mm³)": "producao_gas_nao_associado_mm3",
    "Produção de Água (m³)": "producao_agua_m3",
    "Injeção de Gás (Mm³)": "injecao_gas_mm3",
    "Injeção de Água para Recuperação Secundária (m³)": "injecao_agua_recuperacao_secundaria_m3",
    "Injeção de Água para Descarte (m³)": "injecao_agua_descarte_m3",
    "Injeção de Gás Carbônico (Mm³)": "injecao_gas_carbonico_mm3",
    "Injeção de Nitrogênio (Mm³)": "injecao_nitrogenio_mm3",
    "Injeção de Vapor de Água (t)": "injecao_vapor_agua_t",
    "Injeção de Polímeros (m³)": "injecao_polimeros_m3",
    "Injeção de Outros Fluidos (m³)": "injecao_outros_fluidos_m3"
}

### 4.3.2 Validação da padronização dos nomes

Antes de prosseguir para a padronização dos tipos de dados, o mapeamento definido será aplicado temporariamente às 11 tabelas da camada Bronze.

O objetivo é verificar se as diferenças de nomenclatura existentes entre os períodos são corretamente tratadas e se todas as tabelas passam a apresentar exatamente a mesma estrutura de 21 colunas após a padronização.

Nesta etapa, a transformação será realizada apenas em memória para fins de validação, sem modificar as tabelas armazenadas na camada Bronze.

In [0]:
# Valida se o mapeamento de nomes produz a mesma estrutura
# de 21 colunas em todos os períodos históricos.

tabelas_bronze = [
    "producao_mar_2005_2009",
    "producao_mar_2010_2012",
    "producao_mar_2013_2015",
    "producao_mar_2016_2018",
    "producao_mar_2019",
    "producao_mar_2020",
    "producao_mar_2021",
    "producao_mar_2022",
    "producao_mar_2023",
    "producao_mar_2024",
    "producao_mar_2025"
]

colunas_esperadas = list(mapa_colunas.values())

resultado_padronizacao = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    # Padroniza temporariamente os nomes das colunas,
    # considerando os cabeçalhos com colchetes de 2022 a 2025.
    for coluna_original, coluna_silver in mapa_colunas.items():

        coluna_existente = (
            f"[{coluna_original}]"
            if f"[{coluna_original}]" in df.columns
            else coluna_original
        )

        df = df.withColumnRenamed(
            coluna_existente,
            coluna_silver
        )

    resultado_padronizacao.append(
        (
            tabela,
            len(df.columns),
            df.columns == colunas_esperadas
        )
    )

df_validacao_nomes = spark.createDataFrame(
    resultado_padronizacao,
    [
        "tabela",
        "quantidade_colunas",
        "estrutura_padronizada"
    ]
)

display(df_validacao_nomes)

tabela,quantidade_colunas,estrutura_padronizada
producao_mar_2005_2009,21,true
producao_mar_2010_2012,21,true
producao_mar_2013_2015,21,true
producao_mar_2016_2018,21,true
producao_mar_2019,21,true
producao_mar_2020,21,true
producao_mar_2021,21,true
producao_mar_2022,21,true
producao_mar_2023,21,true
producao_mar_2024,21,true


### 4.3.3 Resultado da padronização dos nomes

A validação demonstrou que o mapeamento definido produz uma estrutura uniforme nas 11 tabelas da camada Bronze.

Após a aplicação temporária da padronização, todas as tabelas permaneceram com **21 atributos** e apresentaram exatamente os mesmos nomes e a mesma ordem de colunas.

O resultado confirma que as diferenças de nomenclatura existentes entre os períodos, incluindo os colchetes presentes nos cabeçalhos de 2022 a 2025, podem ser eliminadas de forma consistente durante a construção da camada Silver.

A padronização dos nomes será, portanto, incorporada ao processo definitivo de transformação antes da consolidação do histórico.

## 4.4 Padronização dos tipos de dados

A avaliação estrutural realizada anteriormente demonstrou que os tipos inferidos durante a ingestão não são uniformes entre os diferentes períodos históricos. Alguns atributos apresentam alternância entre `integer` e `string`, enquanto os arquivos de 2019 a 2021 foram deliberadamente preservados como texto durante a correção de seu processo de parsing.

Na camada Silver, os tipos serão definidos explicitamente de acordo com a natureza de cada atributo.

O atributo `ano` será representado como inteiro, enquanto `mes_ano` será convertido para um tipo de data. Os atributos descritivos serão mantidos como texto e os volumes de produção e injeção serão convertidos para tipos numéricos.

Essa padronização permitirá consolidar os diferentes períodos em uma única estrutura sem depender da inferência automática de tipos.

### 4.4.1 Definição dos tipos da camada Silver

Os atributos serão organizados em grupos de acordo com sua natureza, permitindo aplicar regras de conversão específicas para cada conjunto.

Os atributos quantitativos serão convertidos para `double`, preservando a precisão decimal necessária para representar os volumes registrados pela ANP. Os atributos categóricos serão mantidos como `string`.

In [0]:
# Define os grupos de atributos que receberão regras
# específicas de conversão durante a construção da Silver.

colunas_categoricas = [
    "estado",
    "bacia",
    "campo",
    "poco",
    "ambiente",
    "instalacao"
]

colunas_numericas = [
    "producao_oleo_m3",
    "producao_condensado_m3",
    "producao_gas_associado_mm3",
    "producao_gas_nao_associado_mm3",
    "producao_agua_m3",
    "injecao_gas_mm3",
    "injecao_agua_recuperacao_secundaria_m3",
    "injecao_agua_descarte_m3",
    "injecao_gas_carbonico_mm3",
    "injecao_nitrogenio_mm3",
    "injecao_vapor_agua_t",
    "injecao_polimeros_m3",
    "injecao_outros_fluidos_m3"
]

### 4.4.2 Conversão dos atributos quantitativos

Como os tipos inferidos na camada Bronze variam entre os períodos históricos, a conversão dos atributos quantitativos precisa considerar o tipo atualmente armazenado em cada tabela.

Quando o atributo já estiver representado por um tipo numérico, seu valor será convertido diretamente para `double`. Quando estiver armazenado como `string`, será considerada a convenção numérica presente nos arquivos de origem, removendo o separador de milhares e substituindo a vírgula decimal por ponto antes da conversão.

Essa abordagem evita aplicar tratamentos de texto sobre valores que já foram corretamente interpretados como números durante a ingestão.

Antes da persistência da camada Silver, as conversões serão validadas para verificar se algum valor originalmente preenchido passou a ser nulo em decorrência da transformação.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

# Converte os atributos quantitativos de acordo com o tipo
# existente na Bronze, evitando alterar valores já numéricos.
def converter_para_double(df, coluna):

    tipo_atual = df.schema[coluna].dataType

    if isinstance(tipo_atual, NumericType):
        return F.col(f"`{coluna}`").cast("double")

    return (
        F.regexp_replace(
            F.regexp_replace(
                F.trim(F.col(f"`{coluna}`")),
                r"\.",
                ""
            ),
            ",",
            "."
        )
        .cast("double")
    )

### 4.4.3 Validação das conversões numéricas

Antes da aplicação definitiva das conversões, será verificado se os valores preenchidos na camada Bronze podem ser corretamente convertidos para `double`.

Para cada atributo quantitativo e período histórico, será comparada a existência de valores originalmente não nulos com o resultado obtido após a conversão.

Será considerada uma falha de conversão quando um valor preenchido na Bronze resultar em `null` após sua transformação para o tipo numérico. Essa validação permite identificar possíveis formatos inesperados antes da consolidação do histórico.

In [0]:
# Identifica valores preenchidos na Bronze que não conseguem
# ser convertidos para double pelas regras definidas para a Silver.

resultado_conversao = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    for coluna_original, coluna_silver in mapa_colunas.items():

        coluna_existente = (
            f"[{coluna_original}]"
            if f"[{coluna_original}]" in df.columns
            else coluna_original
        )

        df = df.withColumnRenamed(
            coluna_existente,
            coluna_silver
        )

    for coluna in colunas_numericas:

        valor_convertido = converter_para_double(df, coluna)

        falhas = (
            df
            .filter(
                F.col(f"`{coluna}`").isNotNull()
                & valor_convertido.isNull()
            )
            .count()
        )

        resultado_conversao.append(
            (
                tabela,
                coluna,
                falhas
            )
        )

df_validacao_conversao = spark.createDataFrame(
    resultado_conversao,
    [
        "tabela",
        "atributo",
        "falhas_conversao"
    ]
)

display(
    df_validacao_conversao
    .filter(F.col("falhas_conversao") > 0)
    .orderBy("tabela", "atributo")
)

tabela,atributo,falhas_conversao


### 4.4.4 Resultado da validação dos atributos quantitativos

A validação das conversões não identificou falhas nos atributos quantitativos das 11 tabelas da camada Bronze.

Nenhum valor originalmente preenchido resultou em `null` após a aplicação das regras de conversão para `double`, indicando que os diferentes formatos numéricos existentes no histórico podem ser padronizados de forma consistente.

Dessa forma, as regras definidas para os atributos quantitativos poderão ser utilizadas na construção definitiva da camada Silver.

### 4.4.5 Conversão dos atributos temporais

Além dos atributos quantitativos, a camada Silver deverá padronizar os atributos utilizados para representar o período de referência dos registros.

O atributo `ano` será convertido para `integer`, enquanto `mes_ano`, atualmente representado no formato mês/ano, será convertido para `date`.

Como os dados possuem granularidade mensal, o primeiro dia de cada mês será utilizado como referência na representação da data. Dessa forma, por exemplo, `11/2013` será representado como `2013-11-01`.

Essa transformação não altera o período de referência do registro, mas permite utilizar funções temporais do Spark e facilita a posterior construção da dimensão de tempo na camada Gold.

### 4.4.6 Validação da conversão dos atributos temporais

Antes de aplicar definitivamente a conversão dos atributos temporais na camada Silver, será verificado se os valores existentes nas 11 tabelas Bronze podem ser convertidos para os tipos definidos sem perda de informação.

Para cada período histórico, serão realizadas duas validações:

- o atributo `Ano` será convertido temporariamente para `integer`;
- o atributo `Mês/Ano` será convertido temporariamente para `date`, utilizando o formato `MM/yyyy`.

Será considerada uma falha de conversão quando um valor originalmente preenchido na camada Bronze resultar em `null` após a transformação.

A validação será executada nas 11 tabelas antes da aplicação definitiva das transformações, garantindo que os atributos temporais possam ser padronizados sem perda de valores.

In [0]:
# Verifica se ano e mês/ano podem ser convertidos para os tipos
# definidos para a Silver sem perda de valores preenchidos.

resultado_temporal = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    # Localiza os nomes originais considerando os cabeçalhos
    # com colchetes presentes nos períodos mais recentes.
    coluna_ano = "[Ano]" if "[Ano]" in df.columns else "Ano"
    coluna_mes_ano = "[Mês/Ano]" if "[Mês/Ano]" in df.columns else "Mês/Ano"

    ano_convertido = F.col(f"`{coluna_ano}`").cast("integer")

    mes_ano_convertido = F.to_date(
        F.col(f"`{coluna_mes_ano}`"),
        "MM/yyyy"
    )

    falhas_ano = (
        df
        .filter(
            F.col(f"`{coluna_ano}`").isNotNull()
            & ano_convertido.isNull()
        )
        .count()
    )

    falhas_mes_ano = (
        df
        .filter(
            F.col(f"`{coluna_mes_ano}`").isNotNull()
            & mes_ano_convertido.isNull()
        )
        .count()
    )

    resultado_temporal.append(
        (
            tabela,
            falhas_ano,
            falhas_mes_ano
        )
    )

df_validacao_temporal = spark.createDataFrame(
    resultado_temporal,
    [
        "tabela",
        "falhas_ano",
        "falhas_mes_ano"
    ]
)

display(df_validacao_temporal)

tabela,falhas_ano,falhas_mes_ano
producao_mar_2005_2009,0,0
producao_mar_2010_2012,0,0
producao_mar_2013_2015,0,0
producao_mar_2016_2018,0,0
producao_mar_2019,0,0
producao_mar_2020,0,0
producao_mar_2021,0,0
producao_mar_2022,0,0
producao_mar_2023,0,0
producao_mar_2024,0,0


#### 4.4.6.1 Resultado da validação da conversão dos atributos temporais

A validação demonstrou que os atributos temporais podem ser convertidos para os tipos definidos para a camada Silver sem perda de valores preenchidos.

Nas **11 tabelas da camada Bronze**, não foram identificadas falhas na conversão do atributo `Ano` para `integer` nem na conversão de `Mês/Ano` para `date`.

Todos os períodos apresentaram:

- **0 falhas na conversão de `Ano`**;
- **0 falhas na conversão de `Mês/Ano`**.

O resultado confirma que os formatos utilizados nos atributos temporais são compatíveis com as regras de conversão definidas e que essas transformações poderão ser aplicadas durante a construção definitiva da camada Silver sem introduzir valores nulos decorrentes da conversão.

### 4.4.7 Validação da consistência entre os atributos temporais

Após confirmar que os atributos `Ano` e `Mês/Ano` podem ser convertidos para os tipos definidos sem perda de valores, será avaliada a consistência entre as duas informações temporais.

Para cada registro, o ano informado no atributo `Ano` será comparado ao ano extraído de `Mês/Ano` após sua conversão para `date`.

Será considerada uma inconsistência quando os dois atributos representarem anos diferentes. A validação será realizada nas 11 tabelas Bronze antes da consolidação do histórico, garantindo que as duas referências temporais sejam coerentes entre si.

In [0]:
# Verifica se o ano informado em cada registro corresponde
# ao ano presente no respectivo atributo Mês/Ano.

resultado_consistencia_temporal = []

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    coluna_ano = "[Ano]" if "[Ano]" in df.columns else "Ano"
    coluna_mes_ano = "[Mês/Ano]" if "[Mês/Ano]" in df.columns else "Mês/Ano"

    ano_convertido = F.col(f"`{coluna_ano}`").cast("integer")

    mes_ano_convertido = F.to_date(
        F.col(f"`{coluna_mes_ano}`"),
        "MM/yyyy"
    )

    inconsistencias = (
        df
        .filter(
            ano_convertido != F.year(mes_ano_convertido)
        )
        .count()
    )

    resultado_consistencia_temporal.append(
        (
            tabela,
            inconsistencias
        )
    )

df_consistencia_temporal = spark.createDataFrame(
    resultado_consistencia_temporal,
    [
        "tabela",
        "inconsistencias_temporais"
    ]
)

display(df_consistencia_temporal)

tabela,inconsistencias_temporais
producao_mar_2005_2009,0
producao_mar_2010_2012,0
producao_mar_2013_2015,0
producao_mar_2016_2018,0
producao_mar_2019,0
producao_mar_2020,0
producao_mar_2021,0
producao_mar_2022,0
producao_mar_2023,0
producao_mar_2024,0


#### 4.4.7.1 Resultado da validação da consistência entre os atributos temporais

A comparação entre os atributos `Ano` e `Mês/Ano` não identificou inconsistências em nenhuma das 11 tabelas da camada Bronze.

Em todos os períodos históricos, o ano informado no atributo `Ano` corresponde ao ano extraído de `Mês/Ano`, resultando em **0 inconsistências temporais**.

Esse resultado confirma a coerência entre as duas referências de período existentes nos dados e permite que `ano` seja convertido para `integer` e `mes_ano` para `date` durante a construção da camada Silver sem necessidade de correções adicionais.

### 4.4.8 Resultado da padronização dos tipos de dados

As validações realizadas confirmaram que os tipos definidos para a camada Silver podem ser aplicados de forma consistente às 11 tabelas históricas.

Os **13 atributos quantitativos** podem ser convertidos para `double` sem gerar novos valores nulos. Da mesma forma, o atributo `ano` pode ser convertido para `integer` e `mes_ano` para `date` sem falhas de conversão.

A comparação entre os dois atributos temporais também demonstrou consistência em todo o histórico, não sendo identificados registros em que o ano informado fosse diferente do ano correspondente ao período de referência.

Com essas validações, estão definidas as regras de tipagem que serão utilizadas na construção definitiva da camada Silver:

- `ano`: `integer`;
- `mes_ano`: `date`;
- atributos categóricos: `string`;
- atributos quantitativos de produção e injeção: `double`.

A próxima etapa consiste em aplicar conjuntamente as padronizações de nomenclatura e tipos às 11 tabelas e preparar sua consolidação em uma única estrutura histórica.

## 4.5 Aplicação das transformações

Após a validação das regras de nomenclatura e tipagem, as transformações definidas serão aplicadas às 11 tabelas da camada Bronze.

Para cada período histórico, serão realizadas as seguintes operações:

- padronização dos 21 nomes de colunas;
- conversão de `ano` para `integer`;
- conversão de `mes_ano` para `date`;
- manutenção dos atributos categóricos como `string`;
- conversão dos 13 atributos quantitativos para `double`.

Os DataFrames transformados serão mantidos temporariamente em memória para permitir a validação de sua estrutura antes da consolidação do histórico.

Nesta etapa, nenhuma tabela Silver será persistida. A consolidação e a gravação serão realizadas somente após a confirmação de que todas as tabelas apresentam o mesmo schema e preservam a quantidade de registros existente na camada Bronze.

### 4.5.1 Construção dos DataFrames padronizados

As regras previamente validadas serão aplicadas individualmente a cada uma das 11 tabelas Bronze.

Os DataFrames resultantes serão armazenados em uma estrutura temporária, permitindo que cada período seja validado antes da união dos dados.

Essa estratégia mantém as transformações separadas da persistência e permite identificar eventuais problemas antes da criação da tabela consolidada da camada Silver.

In [0]:
# Aplica as regras validadas de nomenclatura e tipagem
# e mantém cada período temporariamente em memória.

dfs_silver = {}

for tabela in tabelas_bronze:

    df = spark.table(f"workspace.bronze.{tabela}")

    # Padroniza os nomes das 21 colunas.
    for coluna_original, coluna_silver in mapa_colunas.items():

        coluna_existente = (
            f"[{coluna_original}]"
            if f"[{coluna_original}]" in df.columns
            else coluna_original
        )

        df = df.withColumnRenamed(
            coluna_existente,
            coluna_silver
        )

    # Padroniza os atributos temporais.
    df = (
        df
        .withColumn(
            "ano",
            F.col("ano").cast("integer")
        )
        .withColumn(
            "mes_ano",
            F.to_date(F.col("mes_ano"), "MM/yyyy")
        )
    )

    # Garante que os atributos categóricos sejam armazenados como texto.
    for coluna in colunas_categoricas:
        df = df.withColumn(
            coluna,
            F.col(f"`{coluna}`").cast("string")
        )

    # Converte os atributos quantitativos para double,
    # respeitando o tipo existente na respectiva tabela Bronze.
    for coluna in colunas_numericas:
        df = df.withColumn(
            coluna,
            converter_para_double(df, coluna)
        )

    # Mantém explicitamente a mesma ordem de colunas
    # definida para a camada Silver.
    df = df.select(*colunas_esperadas)

    dfs_silver[tabela] = df

### 4.5.2 Validação da estrutura após as transformações

Após a aplicação das transformações, será verificado se os 11 DataFrames apresentam exatamente o mesmo schema.

A validação considerará a quantidade de colunas, seus nomes, sua ordem e seus respectivos tipos de dados.

Essa etapa é necessária antes da consolidação porque a união dos diferentes períodos deve ocorrer somente após a garantia de compatibilidade estrutural entre todos os DataFrames.

In [0]:
# Compara o schema dos 11 DataFrames transformados com o schema
# do primeiro período, utilizado como referência estrutural.

tabela_referencia = tabelas_bronze[0]
schema_referencia = dfs_silver[tabela_referencia].schema

resultado_schemas = []

for tabela, df in dfs_silver.items():

    resultado_schemas.append(
        (
            tabela,
            len(df.columns),
            df.schema == schema_referencia
        )
    )

df_validacao_schemas = spark.createDataFrame(
    resultado_schemas,
    [
        "tabela",
        "quantidade_colunas",
        "schema_padronizado"
    ]
)

display(df_validacao_schemas)

tabela,quantidade_colunas,schema_padronizado
producao_mar_2005_2009,21,true
producao_mar_2010_2012,21,true
producao_mar_2013_2015,21,true
producao_mar_2016_2018,21,true
producao_mar_2019,21,true
producao_mar_2020,21,true
producao_mar_2021,21,true
producao_mar_2022,21,true
producao_mar_2023,21,true
producao_mar_2024,21,true


#### 4.5.2.1 Resultado da validação da estrutura

A validação demonstrou que os 11 DataFrames resultantes das transformações apresentam uma estrutura completamente padronizada.

Todos os períodos permaneceram com **21 atributos** e apresentaram `schema_padronizado = true`, indicando que os nomes, a ordem e os tipos das colunas são iguais em todos os DataFrames.

O resultado confirma que as regras de nomenclatura e tipagem foram aplicadas de forma consistente a todo o histórico e que os diferentes períodos estão estruturalmente compatíveis para posterior consolidação.

Antes da união dos DataFrames, será realizada uma última validação para confirmar que as transformações também preservaram a quantidade de registros existente em cada tabela Bronze.

### 4.5.3 Validação da quantidade de registros

Além da compatibilidade estrutural, é necessário verificar se as transformações aplicadas preservaram integralmente os registros existentes na camada Bronze.

Para cada período histórico, a quantidade de registros da tabela Bronze será comparada à quantidade existente no respectivo DataFrame transformado.

Como as operações realizadas até esta etapa envolvem apenas padronização de nomes e conversão de tipos, não é esperada alteração na quantidade de registros.

A validação permitirá confirmar essa premissa antes da consolidação das 11 tabelas em um único histórico.

In [0]:
# Compara a quantidade de registros da Bronze com os DataFrames
# transformados antes da consolidação do histórico.

resultado_registros = []

for tabela in tabelas_bronze:

    total_bronze = (
        spark.table(f"workspace.bronze.{tabela}")
        .count()
    )

    total_transformado = (
        dfs_silver[tabela]
        .count()
    )

    resultado_registros.append(
        (
            tabela,
            total_bronze,
            total_transformado,
            total_bronze == total_transformado
        )
    )

df_validacao_registros = spark.createDataFrame(
    resultado_registros,
    [
        "tabela",
        "registros_bronze",
        "registros_transformados",
        "registros_preservados"
    ]
)

display(df_validacao_registros)

tabela,registros_bronze,registros_transformados,registros_preservados
producao_mar_2005_2009,60004,60004,true
producao_mar_2010_2012,37103,37103,true
producao_mar_2013_2015,38758,38758,true
producao_mar_2016_2018,56337,56337,true
producao_mar_2019,22173,22173,true
producao_mar_2020,21815,21815,true
producao_mar_2021,21149,21149,true
producao_mar_2022,19225,19225,true
producao_mar_2023,14824,14824,true
producao_mar_2024,15229,15229,true


#### 4.5.3.1 Resultado da validação da quantidade de registros

A comparação entre as tabelas Bronze e os DataFrames transformados demonstrou que a quantidade de registros foi integralmente preservada em todos os períodos históricos.

Nas 11 tabelas analisadas, a quantidade de registros após a aplicação das transformações foi exatamente igual à quantidade existente na respectiva tabela Bronze, resultando em `registros_preservados = true` para todos os períodos.

No total, os DataFrames transformados mantêm os **323.307 registros** existentes no histórico de 2005 a 2025.

Esse resultado confirma que as operações de padronização dos nomes e conversão dos tipos de dados não provocaram inclusão ou exclusão de registros.

Com a estrutura e a quantidade de registros validadas, os 11 DataFrames estão preparados para a consolidação em uma única estrutura histórica.

### 4.5.4 Padronização de valores categóricos

Durante a etapa posterior de modelagem dimensional, foram identificadas variações na representação de alguns valores categóricos que poderiam fazer com que entidades semanticamente equivalentes fossem tratadas como categorias distintas.

A partir dessa identificação, foi realizada uma investigação específica sobre os atributos categóricos da camada Silver, com o objetivo de verificar a ocorrência dessas variações e avaliar a necessidade de padronização antes da construção definitiva do modelo dimensional.

Como a camada Silver tem a função de disponibilizar dados tratados e padronizados, os ajustes identificados nessa investigação foram incorporados a esta etapa de transformação.

In [0]:
# Identifica valores categóricos com caracteres corrompidos
# nos DataFrames transformados antes da consolidação.

for coluna in colunas_categoricas:

    valores_corrompidos = None

    for tabela in tabelas_bronze:
        df_coluna = dfs_silver[tabela].select(coluna)

        if valores_corrompidos is None:
            valores_corrompidos = df_coluna
        else:
            valores_corrompidos = valores_corrompidos.unionByName(df_coluna)

    valores_corrompidos = (
        valores_corrompidos
        .filter(F.col(coluna).contains("�"))
        .distinct()
        .orderBy(coluna)
    )

    if valores_corrompidos.count() > 0:
        print(f"\nColuna: {coluna}")
        display(valores_corrompidos)


Coluna: estado


estado
Cear�
Esp�rito Santo
S�o Paulo



Coluna: bacia


bacia
Cear�
Esp�rito Santo
Rec�ncavo



Coluna: campo


campo
BALEIA AN�
BA�NA
BERBIG�O
BIJUPIR�
B�ZIOS
B�ZIOS_ECO
CANGO�
CURIM�
DOM JO�O
DOM JO�O MAR



Coluna: instalacao


instalacao
Esta��o Coletora de Macau
Esta��o Marap� (Dom Jo�o Mar)
Esta��o Pedra Branca
Esta��o de Paru
FPSO CIDADE DE ITAGUA�
FPSO CIDADE DE ITAJA�
FPSO CIDADE DE MARIC�
FPSO CIDADE DE S�O PAULO
FPSO CIDADE DE VIT�RIA
N�o Informado


#### 4.5.4.1 Resultado da identificação de caracteres corrompidos

A análise identificou valores categóricos contendo o caractere de substituição `�`, indicando problemas de codificação em parte dos dados históricos.

Foram encontradas ocorrências nos atributos `estado`, `bacia`, `campo` e `instalacao`. Os demais atributos categóricos analisados não apresentaram esse caractere.

A comparação com os valores corretamente codificados existentes nos demais períodos históricos permitiu identificar correspondências para as categorias afetadas. Dessa forma, os próprios dados históricos foram utilizados como referência para definir suas representações corretas.

Para evitar alterações indevidas em outros valores categóricos, optou-se pela utilização de um **mapeamento explícito entre os valores corrompidos e suas respectivas representações corretamente codificadas**, restringindo o tratamento exclusivamente às ocorrências identificadas.

A correção será aplicada antes da consolidação definitiva da camada Silver e posteriormente validada para verificar a eliminação das ocorrências do caractere `�` sem perda de registros.

### 4.5.5 Correção dos valores categóricos

Com base nas correspondências identificadas nos próprios dados históricos, será aplicado um mapeamento explícito para corrigir os valores categóricos que apresentam caracteres corrompidos.

A transformação será realizada apenas nos quatro atributos afetados: `estado`, `bacia`, `campo` e `instalacao`.

Além da correção dos caracteres corrompidos, foram padronizadas variações pontuais de acentuação e capitalização identificadas no atributo campo, evitando que diferentes grafias de uma mesma entidade sejam interpretadas como campos distintos.

Os demais atributos permanecerão inalterados e nenhuma linha será removida durante o tratamento.


In [0]:
# Define as correções dos valores categóricos identificados.

correcoes_estado = {
    "Cear�": "Ceará",
    "Esp�rito Santo": "Espírito Santo",
    "S�o Paulo": "São Paulo"
}

correcoes_bacia = {
    "Cear�": "Ceará",
    "Esp�rito Santo": "Espírito Santo",
    "Rec�ncavo": "Recôncavo"
}

correcoes_campo = {
    "BALEIA AN�": "BALEIA ANÃ",
    "BA�NA": "BAÚNA",
    "BERBIG�O": "BERBIGÃO",
    "BIJUPIR�": "BIJUPIRÁ",
    "B�ZIOS": "BÚZIOS",
    "B�ZIOS_ECO": "BÚZIOS_ECO",
    "CANGO�": "CANGOÁ",
    "CURIM�": "CURIMÃ",
    "DOM JO�O": "DOM JOÃO",
    "DOM JO�O MAR": "DOM JOÃO MAR",
    "MARIMB�": "MARIMBÁ",
    "MEXILH�O": "MEXILHÃO",
    "NORDESTE DE SAPINHO�": "NORDESTE DE SAPINHOÁ",
    "NOROESTE DE SAPINHO�": "NOROESTE DE SAPINHOÁ",
    "PERO�": "PEROÁ",
    "SAPINHO�": "SAPINHOÁ",
    "SUDOESTE DE SAPINHO�": "SUDOESTE DE SAPINHOÁ",
    "S�PIA": "SÉPIA",
    "S�PIA LESTE": "SÉPIA LESTE",
    "TAMBA�": "TAMBAÚ",
    "TAMBUAT�": "TAMBUATÁ",
    "TUBAR�O AZUL": "TUBARÃO AZUL",
    "TUBAR�O MARTELO": "TUBARÃO MARTELO",
    "URUGU�": "URUGUÁ",
    "XAR�U": "XARÉU",
    
    # Padroniza variações adicionais de nomenclatura identificadas
    "NOROESTE DE SAPINHOA": "NOROESTE DE SAPINHOÁ",
    "AnC_BRAVA": "ANC_BRAVA",
    "AnC_NORTE_ATAPU": "ANC_NORTE_ATAPU",
    "AnC_TUPI": "ANC_TUPI"
}

correcoes_instalacao = {
    "Esta��o Coletora de Macau": "Estação Coletora de Macau",
    "Esta��o Marap� (Dom Jo�o Mar)": "Estação Marapé (Dom João Mar)",
    "Esta��o Pedra Branca": "Estação Pedra Branca",
    "Esta��o de Paru": "Estação de Paru",
    "FPSO CIDADE DE ITAGUA�": "FPSO CIDADE DE ITAGUAÍ",
    "FPSO CIDADE DE ITAJA�": "FPSO CIDADE DE ITAJAÍ",
    "FPSO CIDADE DE MARIC�": "FPSO CIDADE DE MARICÁ",
    "FPSO CIDADE DE S�O PAULO": "FPSO CIDADE DE SÃO PAULO",
    "FPSO CIDADE DE VIT�RIA": "FPSO CIDADE DE VITÓRIA",
    "N�o Informado": "Não Informado",
    "PLATAFORMA DE CURIM� 1": "PLATAFORMA DE CURIMÃ 1",
    "PLATAFORMA DE CURIM� 2": "PLATAFORMA DE CURIMÃ 2",
    "Plataforma de Mexilh�o": "Plataforma de Mexilhão",

    # Variações de capitalização
    "FPSO Fluminense": "FPSO FLUMINENSE",
    "Peregrino A": "PEREGRINO A",
    "Peregrino B": "PEREGRINO B",
    "Plataforma de Mexilhão": "PLATAFORMA DE MEXILHÃO",
    "Petrobras 18": "PETROBRAS 18",
    "Petrobras 19": "PETROBRAS 19",
    "Petrobras 20": "PETROBRAS 20",
    "Petrobras 25": "PETROBRAS 25",
    "Petrobras 31": "PETROBRAS 31",
    "Petrobras 35": "PETROBRAS 35",
    "Petrobras 40": "PETROBRAS 40",
    "Petrobras 43": "PETROBRAS 43",
    "Petrobras 48": "PETROBRAS 48",
    "Petrobras 51": "PETROBRAS 51",
    "Petrobras 52": "PETROBRAS 52",
    "Petrobras 53": "PETROBRAS 53",
    "Petrobras 54": "PETROBRAS 54",
    "Petrobras 55": "PETROBRAS 55",
    "Petrobras 56": "PETROBRAS 56",
    "Petrobras 57": "PETROBRAS 57",
    "Petrobras 58": "PETROBRAS 58",
    "Petrobras 62": "PETROBRAS 62",
    "Petrobras 66": "PETROBRAS 66",
    "Petrobras 67": "PETROBRAS 67",
    "Petrobras 68": "PETROBRAS 68",
    "Petrobras 69": "PETROBRAS 69",
    "Petrobras 70": "PETROBRAS 70",
    "Petrobras 74": "PETROBRAS 74",
    "Petrobras 75": "PETROBRAS 75",
    "Petrobras 76": "PETROBRAS 76",
    "Petrobras 77": "PETROBRAS 77",

    # Variações de espaçamento, hífen e grafia
    "PLATAFORMA DE PAMPO-1": "PLATAFORMA DE PAMPO 1",
    "PLATAFORMA DE PARGO-1A": "PLATAFORMA DE PARGO 1A",
    "FPSO CIDADE DE ILHA BELA": "FPSO CIDADE DE ILHABELA",
    "BRAVA SATAR": "BRAVA STAR",

    # Numeração romana e arábica
    "PLATAFORMA DE CARAPEBA-I": "PLATAFORMA DE CARAPEBA 1",
    "PLATAFORMA DE CARAPEBA-II": "PLATAFORMA DE CARAPEBA 2",
    "PLATAFORMA DE CARAPEBA-III": "PLATAFORMA DE CARAPEBA 3",
    "PLATAFORMA DE VERMELHO-I": "PLATAFORMA DE VERMELHO 1",
    "PLATAFORMA DE VERMELHO-III": "PLATAFORMA DE VERMELHO 3",

    # Variações com a sigla da plataforma entre parênteses
    "Petrobras 66 (P-66)": "PETROBRAS 66",
    "Petrobras 67 (P-67)": "PETROBRAS 67",
    "Petrobras 74 (P-74)": "PETROBRAS 74",
    "Petrobras 75 (P-75)": "PETROBRAS 75"
}

# Associa cada atributo ao seu respectivo mapeamento.
correcoes_por_coluna = {
    "estado": correcoes_estado,
    "bacia": correcoes_bacia,
    "campo": correcoes_campo,
    "instalacao": correcoes_instalacao
}

# Aplica as correções nos 11 DataFrames que serão
# posteriormente consolidados na camada Silver.
for tabela in tabelas_bronze:
    for coluna, correcoes in correcoes_por_coluna.items():
        dfs_silver[tabela] = (
            dfs_silver[tabela]
            .replace(correcoes, subset=[coluna])
        )

### 4.5.6 Validação da correção dos valores categóricos

Após a aplicação do mapeamento, será verificado se ainda existem valores contendo o caractere `�` nos atributos categóricos.

Também será validada novamente a quantidade total de registros dos 11 DataFrames transformados, garantindo que a padronização dos valores não tenha provocado perda ou exclusão de linhas.

A correção será considerada válida caso não sejam encontradas ocorrências do caractere corrompido e o total permaneça igual aos 323.307 registros existentes antes do tratamento.

In [0]:
# Valida a ausência de caracteres corrompidos e confirma
# a preservação da quantidade de registros após as correções.

resultado_validacao = []

for coluna in colunas_categoricas:

    ocorrencias_corrompidas = sum(
        dfs_silver[tabela]
        .filter(F.col(coluna).contains("�"))
        .count()
        for tabela in tabelas_bronze
    )

    resultado_validacao.append(
        (coluna, ocorrencias_corrompidas)
    )

df_validacao_caracteres = spark.createDataFrame(
    resultado_validacao,
    ["atributo", "ocorrencias_com_caractere_corrompido"]
)

display(df_validacao_caracteres)

# Valida novamente a quantidade total de registros.
total_registros_apos_correcao = sum(
    dfs_silver[tabela].count()
    for tabela in tabelas_bronze
)

print(
    f"Total de registros após a correção: "
    f"{total_registros_apos_correcao}"
)

atributo,ocorrencias_com_caractere_corrompido
estado,0
bacia,0
campo,0
poco,0
ambiente,0
instalacao,0


Total de registros após a correção: 323307


#### 4.5.6.1 Resultado da validação da correção dos valores categóricos

A validação realizada após a padronização dos valores categóricos não identificou ocorrências do caractere `�` em nenhum dos seis atributos categóricos analisados.

Os atributos `estado`, `bacia`, `campo`, `poco`, `ambiente` e `instalacao` apresentaram zero ocorrências do caractere corrompido após o tratamento.

Também foi confirmada a preservação da quantidade de registros, permanecendo o total de **323.307 linhas** nos 11 DataFrames transformados.

Os resultados demonstram que a correção dos valores categóricos foi realizada sem perda de registros e eliminou as inconsistências de codificação identificadas durante a etapa de modelagem dimensional.

Com essa validação, os DataFrames estão preparados para serem consolidados e persistidos na camada Silver.

## 4.6 Consolidação do histórico

Após a padronização e validação individual dos 11 DataFrames, os diferentes períodos históricos serão consolidados em uma única estrutura.

Como todos os DataFrames apresentam os mesmos 21 atributos, na mesma ordem e com os mesmos tipos de dados, a consolidação poderá ser realizada por meio da união dos registros.

O resultado deverá conter todo o histórico de produção offshore compreendido entre 2005 e 2025, preservando os **323.307 registros** provenientes da camada Bronze.

Inicialmente, a consolidação será realizada em memória. Antes da persistência da tabela Silver, serão validadas a quantidade total de registros e a cobertura temporal do conjunto consolidado.

### 4.6.1 União dos períodos históricos

Os 11 DataFrames transformados serão unidos sequencialmente em um único DataFrame denominado `df_silver`.

Será utilizado `unionByName`, realizando a correspondência dos atributos pelos nomes das colunas. Embora a ordem e o schema já tenham sido previamente validados, essa abordagem torna explícita a correspondência entre os atributos durante a consolidação.

Nenhuma agregação ou remoção de registros será realizada nesta etapa. O objetivo é apenas reunir os diferentes períodos em uma única estrutura histórica.

In [0]:
from functools import reduce

# Consolida os 11 períodos históricos utilizando os nomes
# padronizados das colunas como referência para a união.

df_silver = reduce(
    lambda df1, df2: df1.unionByName(df2),
    [dfs_silver[tabela] for tabela in tabelas_bronze]
)


### 4.6.2 Validação da consolidação

Após a união dos períodos, será realizada uma validação do conjunto histórico consolidado.

Serão verificados:

- a quantidade total de registros;
- o menor e o maior ano presentes no histórico;
- a menor e a maior data de referência;
- a quantidade de anos distintos;
- a quantidade de meses distintos.

Essas verificações permitem confirmar se a consolidação preservou o volume esperado de dados e se o período de 2005 a 2025 está integralmente representado antes da persistência da camada Silver.

In [0]:
# Valida o volume e a cobertura temporal do histórico consolidado
# antes de sua persistência na camada Silver.

df_validacao_consolidacao = (
    df_silver
    .agg(
        F.count("*").alias("total_registros"),
        F.min("ano").alias("ano_inicial"),
        F.max("ano").alias("ano_final"),
        F.min("mes_ano").alias("mes_inicial"),
        F.max("mes_ano").alias("mes_final"),
        F.countDistinct("ano").alias("anos_distintos"),
        F.countDistinct("mes_ano").alias("meses_distintos")
    )
)

display(df_validacao_consolidacao)

total_registros,ano_inicial,ano_final,mes_inicial,mes_final,anos_distintos,meses_distintos
323307,2005,2025,2005-01-01,2025-12-01,21,252


#### 4.6.2.1 Resultado da validação da consolidação

A validação do DataFrame consolidado confirmou a preservação do volume e da cobertura temporal esperados para o histórico.

O conjunto resultante contém **323.307 registros**, correspondendo à mesma quantidade total identificada anteriormente nas 11 tabelas da camada Bronze e nos DataFrames transformados.

A cobertura temporal compreende o período de **janeiro de 2005 a dezembro de 2025**, totalizando:

- **21 anos distintos**;
- **252 meses distintos**;
- primeiro período de referência em **2005-01-01**;
- último período de referência em **2025-12-01**.

Como o intervalo de 21 anos corresponde a 252 meses e foram identificadas 252 referências mensais distintas, o resultado confirma que **todos os meses compreendidos entre janeiro de 2005 e dezembro de 2025 estão representados no conjunto consolidado**.

Dessa forma, a consolidação das 11 tabelas preservou a quantidade de registros e a cobertura temporal do histórico, permitindo prosseguir para a persistência da tabela na camada Silver.

## 4.7 Persistência da camada Silver

Após a validação das transformações e da consolidação do histórico, o DataFrame resultante será persistido no schema `silver` do catálogo `workspace`.

A tabela será armazenada em formato **Delta**, mantendo a arquitetura Lakehouse adotada no projeto e permitindo que os dados tratados sejam utilizados de forma estruturada pelas etapas posteriores de modelagem e análise.

Será criada uma única tabela consolidada denominada `producao_mar`, contendo o histórico padronizado da produção offshore entre 2005 e 2025.

### 4.7.1 Criação da tabela Silver

O DataFrame consolidado será persistido como a tabela `workspace.silver.producao_mar`.

Será utilizado o modo `overwrite` para permitir a reexecução controlada do notebook durante o desenvolvimento do projeto. Como o schema da tabela já foi definido e validado nas etapas anteriores, a gravação utilizará a estrutura padronizada construída para a camada Silver.

In [0]:
# Persiste o histórico tratado e consolidado como tabela Delta
# na camada Silver do Lakehouse.

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.producao_mar")
)

## 4.8 Validação da tabela Silver

Após a persistência, a tabela Silver será consultada novamente diretamente do catálogo para confirmar que o objeto gravado apresenta as características esperadas.

Essa validação é realizada sobre a tabela persistida, e não sobre o DataFrame temporário utilizado em sua construção.

Serão verificados inicialmente:

- quantidade total de registros;
- quantidade de colunas;
- ano inicial e final;
- quantidade de anos distintos;
- quantidade de meses distintos.

Essa etapa permite confirmar que a persistência em formato Delta não alterou a estrutura ou a cobertura dos dados consolidados.

In [0]:
# Lê novamente a tabela diretamente da camada Silver para validar
# o objeto efetivamente persistido no catálogo.

df_silver_persistida = spark.table(
    "workspace.silver.producao_mar"
)

df_validacao_silver = (
    df_silver_persistida
    .agg(
        F.count("*").alias("total_registros"),
        F.min("ano").alias("ano_inicial"),
        F.max("ano").alias("ano_final"),
        F.countDistinct("ano").alias("anos_distintos"),
        F.countDistinct("mes_ano").alias("meses_distintos")
    )
    .withColumn(
        "quantidade_colunas",
        F.lit(len(df_silver_persistida.columns))
    )
)

display(df_validacao_silver)

total_registros,ano_inicial,ano_final,anos_distintos,meses_distintos,quantidade_colunas
323307,2005,2025,21,252,21


### 4.8.1 Resultado da validação da tabela persistida

A validação realizada diretamente sobre a tabela `workspace.silver.producao_mar` confirmou que a persistência em formato Delta preservou as características do conjunto consolidado.

A tabela Silver contém **323.307 registros e 21 atributos**, mantendo integralmente o volume de dados obtido após a consolidação das 11 tabelas da camada Bronze.

A cobertura temporal também foi preservada, compreendendo:

- **21 anos distintos**, entre 2005 e 2025;
- **252 meses distintos**, correspondentes ao período completo de janeiro de 2005 a dezembro de 2025.

Os resultados são consistentes com as validações realizadas antes da persistência, indicando que a gravação da tabela Delta não provocou alteração na quantidade de registros, na estrutura ou na cobertura temporal do histórico.

### 4.8.2 Validação do schema persistido

Como etapa final de validação da camada Silver, será inspecionado o schema da tabela persistida no catálogo.

O objetivo é confirmar que os tipos definidos durante a transformação foram mantidos após a gravação em formato Delta.

Espera-se que `ano` esteja armazenado como `integer`, `mes_ano` como `date`, os atributos categóricos como `string` e os 13 atributos quantitativos como `double`.

In [0]:
# Inspeciona o schema da tabela Silver persistida para confirmar
# a manutenção dos tipos definidos durante a transformação.

df_schema_silver = spark.createDataFrame(
    [
        (campo.name, campo.dataType.simpleString())
        for campo in df_silver_persistida.schema.fields
    ],
    ["atributo", "tipo"]
)

display(df_schema_silver)

atributo,tipo
ano,int
mes_ano,date
estado,string
bacia,string
campo,string
poco,string
ambiente,string
instalacao,string
producao_oleo_m3,double
producao_condensado_m3,double


#### 4.8.2.1 Resultado da validação do schema persistido

A inspeção do schema da tabela `workspace.silver.producao_mar` confirmou que os tipos definidos durante a transformação foram preservados após a persistência em formato Delta.

A estrutura final apresenta:

- `ano` como `integer`;
- `mes_ano` como `date`;
- os seis atributos categóricos como `string`;
- os treze atributos quantitativos de produção e injeção como `double`.

O resultado confirma que a tabela persistida apresenta uma estrutura uniforme e compatível com as necessidades das etapas posteriores de modelagem e análise.

Dessa forma, as diferenças de nomenclatura e tipagem existentes entre os arquivos históricos foram eliminadas na camada Silver sem alterar as tabelas originais da camada Bronze.

## 4.9 Conclusão da criação da camada Silver

A construção da camada Silver consolidou os dados históricos de produção offshore da ANP entre 2005 e 2025 em uma única estrutura padronizada.

A partir das regras definidas durante a avaliação de qualidade, foram realizadas a padronização dos nomes das colunas, a definição explícita dos tipos de dados, a conversão dos atributos quantitativos e temporais e a integração dos 11 períodos históricos.

As validações realizadas ao longo do processo confirmaram que:

- as 11 tabelas foram padronizadas para a mesma estrutura de 21 atributos;
- as conversões dos atributos quantitativos e temporais foram realizadas sem falhas;
- a consistência entre `ano` e `mes_ano` foi preservada;
- nenhum registro foi perdido durante as transformações;
- o histórico consolidado contém **323.307 registros**;
- o período completo entre **janeiro de 2005 e dezembro de 2025** está representado, totalizando **252 meses distintos**;
- o schema definido foi preservado após a persistência em formato Delta.

Como resultado, foi criada a tabela `workspace.silver.producao_mar`, que representa a fonte tratada e consolidada do projeto.

A partir desta estrutura, a próxima etapa consiste na definição do modelo dimensional que orientará posteriormente a construção da camada Gold e a organização dos dados de acordo com as necessidades das perguntas de negócio.